# 04 - Feature extraction and selection

**Aim:** extract N-terminal protein features and evaluate Random-Forest feature selection within nested cross-validation.

We planned this section around the representative sequences and saved folds from 02b. The composition analysis in 02c helped us choose candidate features, and we used 03 as the reference for loading data and reporting results. We will compare RF-selected features with all features using the same fixed SVM and outer folds, keeping benchmarking for later evaluation.

**Materials**
- `04 - Methods - Feature Extraction and Selection.pdf` (slides 11–19, 27–30, 34–48 and 50–60).
- `04_worked_example_nested_cv.ipynb` (sections 2, 5–6, 8–10 and 12), the updated Colab example linked from slide 49.
- `03_von_heijne_method.ipynb` (loading in 1.1 and reporting in section 5); `02c_data_analysis.ipynb` (N-terminal composition in section 5).

**Status:** we prepared this notebook as a work guide and left the code cells empty. Implementation, execution, figures and observations are still pending. The example references below refer to `04_worked_example_nested_cv.ipynb`, which uses synthetic data rather than our proteins.

## Work distribution and handoffs

We defined the following work distribution and shared interfaces to guide the implementation. Each person will implement one consecutive block, add its brief explanation and pass the cumulative notebook to the next person. We assigned final integration to Person 5.

| Object | Shared interface |
| --- | --- |
| Feature blocks | Numeric DataFrames indexed by `accession`: `composition_features`, `hydropathy_features`, `property_features`. |
| `X`, `y`, `cv_fold` | Training feature matrix, labels and saved folds, aligned to the same accession order. |
| Feature catalog | Columns `feature`, `definition`, `region`, `scale`, `window`, `aggregation`, `unit`. |
| RF selector | Fit on training rows only; retain selected names and transform other rows using the same columns and order. |
| `cv_predictions` | `method`, `accession`, `test_fold`, `label`, `prediction`; methods `all_features` and `rf_top_k`. |
| `cv_per_fold` | `method`, `test_fold`, `selected_k`, `precision`, `recall`, `accuracy`, `mcc`, `f1`. |
| `inner_scores` | `test_fold`, `k`, `mean_validation_mcc`. |
| `selected_features` | `test_fold`, `feature`, `importance`, `rank`, for features retained after the outer-training refit. |

We arranged the work as a sequence of handoffs: **Person 1 → Person 2 → Person 3 → Person 4 → Person 5**. Each person will start after receiving the completed preceding block. Everyone can read their slides and example references in advance.

| Person | Consecutive block | Receives | Passes to the next person |
| --- | --- | --- | --- |
| 1 | Sections 0–1: data and definitions | Prepared FASTA/TSV and course materials. | To Person 2: cumulative notebook with `data`, reserved `benchmarking_data`, paths, region/alphabet/window settings and documented property mappings. |
| 2 | Section 2: all feature extraction | Person 1's data and definitions. | To Person 3: cumulative notebook with the three accession-indexed feature blocks, extraction functions and catalog entries. |
| 3 | Sections 3–4: matrix and selection setup | Person 2's feature blocks and definitions. | To Person 4: cumulative notebook with saved raw features/catalog, aligned `X`, `y`, `cv_fold`, unfitted RF/SVM and baseline pipelines, and the `k` grid. |
| 4 | Section 5: nested CV and baseline evaluation | Person 3's matrix and unfitted pipelines. | To Person 5: cumulative notebook with `cv_predictions`, `cv_per_fold`, `inner_scores` and `selected_features`, plus completed CV explanations. |
| 5 | Section 6: results and final integration | Person 4's results and all preceding contributions. | To the group: integrated notebook, exported tables/figures and observations based on real results. |

We included scale selection in Person 1's preparation so Person 2 can begin extraction with the required mappings. We assigned both pipelines to Person 3, fitting and evaluation to Person 4, and final tables and figures to Person 5. This keeps each contribution in one consecutive block.

**Similar code:** the example section 2 defines selector and result interfaces; section 12 connects selection to a pipeline. Adapt these interfaces to accessions and saved folds.

## 0. Data loading

We kept the path convention from 03: paths are relative to `notebooks/`. We also chose the same Conda `LB2` environment and notebook kernel.

### 0.1 Imports and paths

**Pending — Person 1.** Prepare imports for data handling, plotting, RF, SVM, pipelines, grid search and metrics. Define `input_dir` as `../data/prepared`, `feature_dir` as `../data/features/04`, `table_dir` as `../results/tables/04` and `figure_dir` as `../results/figures/04`.

**Output:** shared imports and path variables; create output directories when implementation starts.

**Similar code:** the example's opening imports and section 12 cover the ML libraries. For relative paths and PDF export conventions, follow 03 section 0.

### 0.2 Load sequences and metadata

**Pending — Person 1.** Read `positive_dataset.tsv`, `negative_dataset.tsv` and their matching FASTA files. Associate sequences by `accession`, add `label` (1 for SP, 0 for NO_SP), and concatenate into `prepared_data`. Keep training rows in `data` and the reserved split in `benchmarking_data`; retain `cv_fold` as nullable integer.

**Output:** protein tables with sequence, label and original metadata. Training contains 881 positives and 7,265 negatives; benchmarking contains 221 positives and 1,817 negatives. Preserve the saved splits and folds 1–5.

**Similar code:** the example section 1 generates synthetic data, so it has no equivalent protein loader. Reuse `read_fasta` and accession mapping from 03 section 1.1, keeping this notebook independent of other kernels.

## 1. Feature definition

We chose the same N-terminal region for both classes because it can be extracted from any protein sequence without knowing its cleavage site (slides 11–19 and 27–30).

### 1.1 Region and common settings

**Pending — Person 1.** Define `n_terminal_length = 90` and use the available residues for shorter proteins. We chose 90 residues to continue the approach in 03; the 70-residue comparison in 02c was exploratory. Use complete 9-position windows for the hydropathy profile.

We agreed to express composition as canonical-residue fractions. For property profiles, we will preserve sequence positions, mark non-canonical values as missing and average known values within each original window. Person 1 will document the Kyte–Doolittle mapping and choose one charge-related scale and one residue-volume scale from the sources in slides 18–19. Record all mappings, sources, units and any pH assumption for charge before handing the notebook to Person 2. We will keep these settings fixed during evaluation.

**Output:** shared region, alphabet, window size and documented scale definitions for Person 2. The numerical charge and volume mappings are still pending implementation by Person 1.

**Similar code:** the example has no protein-region or property-profile extractor. Use the sequence slicing in 03 section 3 and canonical-residue counting in 02c section 5 as partial references.

## 2. Feature extraction

We chose composition and physicochemical properties to describe each training protein with a fixed-size vector (slides 15–19 and 27–30). The planned extraction uses only sequence information, so it does not require labels or model fitting.

### 2.1 Amino-acid composition

**Pending — Person 2; receives the completed sections 0–1 from Person 1.** Calculate the fraction of each of the 20 canonical amino acids in the first 90 residues of each protein. Exclude non-canonical residues from the denominator. Use names `aa_fraction_A` through `aa_fraction_Y`, in alphabet order `ACDEFGHIKLMNPQRSTVWY`.

**Output:** `composition_features` and corresponding catalog entries, with fractions on a 0–1 scale. The calculation needs a separate row for each protein; 02c pooled residues within each class.

**Similar code:** none in the example. Adapt canonical counting from 02c section 5.2; that code pools residues and therefore needs a per-protein adaptation.

### 2.2 Hydropathy

**Pending — Person 2; receives the completed sections 0–1 from Person 1.** Use the Kyte–Doolittle mapping supplied by Person 1 in 1.1. Map each N-terminal residue to its hydropathy value, calculate its global mean and the maximum mean over complete 9-position windows. Keep unknown positions in place as agreed in 1.1.

**Output:** `hydropathy_features` with `hydropathy_mean` and `hydropathy_max_w9`, plus scale source and catalog entries. Define reusable extraction functions before applying them.

**Similar code:** none in the example. Slides 16–19 illustrate property profiles and sources; slides 27–30 motivate mean/max aggregation and local windows.

### 2.3 Charge and volume

**Pending — Person 2; receives the completed sections 0–1 from Person 1.** Use the charge-related and residue-volume mappings supplied by Person 1 in 1.1, retaining their documented units and assumptions. Calculate the mean known value in the same N-terminal region.

**Output:** `property_features` with `charge_mean` and `volume_mean`, plus catalog entries. Describe the chosen charge scale and its assumptions. The groups in 02c helped organize the plots but did not measure numerical charge.

**Similar code:** none in the example. Use the profile-and-aggregation approach from 2.2 with these separately documented scales (slides 15–19 and 27–29).

## 3. Feature matrix

We planned to combine the three feature blocks into one matrix, with a row for each training protein (slides 34–35).

### 3.1 Assemble predictors and aligned labels

**Pending — Person 3; receives all three completed feature blocks from Person 2.** Join the three blocks by accession in the training-table order. Use the 20 composition columns first, then hydropathy, charge and volume. Build `X`, `y`, `cv_fold` and the feature catalog using the shared interfaces in the opening work-distribution note.

**Output:** the training feature matrix and aligned labels/folds. Keep `cleavage_pos`, `has_tm_in_90`, organism, kingdom, length, labels and split/fold metadata outside `X`. Do not add a globally trained Von Heijne score.

**Similar code:** the example sections 1–2 align `X`, `y` and `feature_names`, but do not join protein feature blocks. Accession-based alignment follows 03 section 1.1.

### 3.2 Save extracted features

**Pending — Person 3.** Save the raw, unscaled training matrix with accession as its index to `feature_dir / training_features.tsv`. Save definitions to `feature_dir / feature_catalog.tsv`. Labels and folds remain available from the prepared metadata.

**Output:** two stage-specific TSV files. Save no global feature ranking or globally scaled matrix; those fitted quantities belong inside CV.

**Similar code:** no feature-file export in the example. Follow the TSV export style in 03 section 5.5 and use the paths defined in 0.1.

## 4. Feature selection

We chose RF importance to rank features and retain the top-k (slides 42–44). To keep test proteins out of these decisions, we will fit selection and scaling within each training fold (slides 38 and 59).

### 4.1 RF top-k selector

**Pending — Person 3.** Adapt RF fitting, Gini importance ranking and top-k selection to a fitted selector that can be included in a pipeline. Start with the example's fixed settings: 100 trees and `random_state = 0`. Expose `k` for the inner search and retain feature names, ranks and importances after fitting.

**Output:** an unfitted selector supporting training-only fitting and transformation with the fitted column order. Include an option to retain all columns.

**Similar code:** example section 5, `select_rf`, gives the RF ranking and slicing logic. It returns column indices; adapting that function to a pipeline-compatible fitted selector remains to be implemented.

### 4.2 Pipeline and selection settings

**Pending — Person 3.** Connect RF selection, `StandardScaler` and a fixed RBF SVM, in that order. Use `C = 1.0` and `gamma = scale`, as in example section 10. Person 3 will also prepare the baseline pipeline with the same scaler/SVM and all features.

Use candidates 5, 10, 15, 20 and the full feature count, removing duplicates and values exceeding the number of columns. We chose mean inner-validation MCC because the classes are imbalanced, adapting the example's accuracy criterion. Search only `k`, leaving the other settings fixed.

**Output:** unfitted selected-feature and baseline pipelines, plus the `k` grid. For tied mean scores, retain the smallest candidate `k`.

**Similar code:** example section 12 provides `Pipeline` and `GridSearchCV`, but uses mutual information and logistic regression. Combine that structure with `select_rf` from section 5 and `new_svm` from section 10; the RF pipeline is not provided directly.

## 5. Nested cross-validation

We chose the nested-CV protocol from slides 45–48. For each outer test fold, we will use the other four saved folds for training. Each will serve as inner validation once, while the remaining three train the selector, scaler and classifier. This extends the single cyclic validation fold used in 03.

### 5.1 Reuse the saved folds

**Pending — Person 4; receives the completed matrix and pipeline setup from Person 3.** Prepare five outer train/test index pairs from `cv_fold`. Within each outer-training subset, prepare four inner train/validation pairs from the remaining saved fold identifiers. Use positions within the outer-training subset for the inner indices.

**Output:** outer indices and the corresponding inner index lists for the inner search and outer evaluation in 5.2–5.4. Do not reshuffle, repartition or include benchmarking.

**Similar code:** example section 2 shows outer/inner loops with `StratifiedKFold`; section 12 accepts CV specifications. Replace the generated splits with indices from our saved folds. The cyclic `get_cv_split` helper from 03 is not the protocol for 04.

### 5.2 Choose k in the inner loop

**Pending — Person 4.** For each outer-training subset, use grid search with its four inner index pairs and MCC scoring. Refit selector, scaler and SVM within every inner training split. Record each candidate's mean validation MCC and the selected `k`.

**Output:** `inner_scores` and the best fitted pipeline for each outer round, refitted on all four outer-training folds. The outer test fold must not influence any choice.

**Similar code:** example section 2, `choose_setting`, explains the inner loop; section 12, `inner_search`, provides the grid-search pattern. Adapt the scoring criterion, selector and saved-fold indices.

### 5.3 Evaluate outer test predictions

**Pending — Person 4.** Use each refitted pipeline to predict its outer test fold once. Record predictions and selected-feature names/importances using the shared interfaces in the opening work-distribution note. Calculate precision, recall, accuracy, MCC and F1 from test labels and predictions.

**Output:** `cv_predictions`, RF rows of `cv_per_fold`, and `selected_features`, covering each training protein once as an outer test case. No test-based threshold or parameter adjustment is planned.

**Similar code:** example section 2, `nested_cv`, shows refit, test and selected-set recording; section 12 shows outer scoring. Use an explicit outer loop around the inner grid search to retain predictions and selections. Adapt metric calculations from 03 section 5 rather than copying the example's accuracy-only return.

### 5.4 Evaluate the all-feature baseline

**Pending — Person 4.** Fit the baseline scaler/SVM on each four-fold outer-training subset and predict the same external fold. Its parameters are fixed, so no inner search is needed. Append baseline predictions and metrics using `method = all_features`; set `selected_k` to the full feature count.

**Output:** baseline rows in the same result tables, allowing a paired comparison with `rf_top_k` on identical test proteins.

**Similar code:** example section 2 provides `select_all` and the no-selection baseline; section 10 compares the same SVM with all features and RF top-k.

## 6. Results and interpretation

We planned the reporting around outer-test performance and variation in feature selection, following the table and numbered-PDF conventions in 03. Results and observations are still pending.

### 6.1 Metrics and paired comparison

**Pending — Person 5; receives the completed CV results from Person 4.** Summarize each method's five outer-test metrics as mean and standard deviation. Use `ddof = 0` to match the worked example, and label the spread as SD; 03 reported SE. Store `method`, `metric`, `mean` and `std` in `cv_summary`. Compare RF top-k minus baseline within each matching test fold.

**Output:** `table_dir / 6_1_cv_per_fold.tsv`, `table_dir / 6_1_cv_summary.tsv` and `table_dir / 6_1_cv_predictions.tsv`. Show the paired metrics and feature counts without inventing values.

**Similar code:** example section 6 builds a comparison table; section 7 shows paired differences. Adapt these to our five fixed folds and metrics, without repeating or reshuffling CV. The synthetic 'signal/noise kept' columns have no known ground truth here.

### 6.2 Selection frequency

**Pending — Person 5.** Count how often each feature is retained across the five outer-training refits. Include unselected features with zero counts. Report `feature`, `selected_folds` and `selection_fraction`; use the feature catalog to explain the most frequently retained characteristics.

**Output:** `table_dir / 6_2_selected_features.tsv`, `table_dir / 6_2_selection_frequency.tsv` and a bar plot saved as `figure_dir / 6_2_selection_frequency.pdf`.

**Similar code:** example section 9 counts and plots selection frequency across methods/folds. Adapt to one selector and five refits; do not use synthetic informative/redundant/noise colors or interpret a single feature list as a biological finding (slide 57).

### 6.3 Inner-validation curves

**Pending — Person 5.** Plot mean inner-validation MCC against `k` for the five outer rounds and mark the selected values. Treat each peak as a selection score, not as the performance estimate.

**Output:** `table_dir / 6_3_inner_scores.tsv` and `figure_dir / 6_3_inner_mcc_vs_k.pdf`. Save the PDF, print its relative path and display the plot, as in 03.

**Similar code:** example section 8 plots inner curves for RFE. Reuse the plot structure for RF top-k and MCC; slides 53–54 explain how to read the curves.

### 6.4 Observations and integration

**Pending — Person 5.** Summarize whether RF selection changes test performance relative to the all-feature SVM, how many features are retained, and how selection varies across folds. Discuss gains alongside their fold-to-fold spread without claiming statistical significance from five folds alone.

Use `../results/tables/03/5_4_cv_per_round.tsv` and `../results/tables/03/5_5_cv_summary.tsv` only for a descriptive reference to the completed Von Heijne analysis. We chose different training protocols for the two methods, so this comparison will be descriptive. Label 03's uncertainty as SE and 04's as SD. We will keep benchmarking for later evaluation.

**Deliverable:** concise observations grounded in saved results and a notebook assembled in scientific order. Each person will add the explanation for their contribution before handing it on. Person 5 will bring the contributions together and keep names, paths and dependencies consistent. Replace pending instructions with completed prose only after the corresponding work is done.

**Similar code:** example sections 6, 8–9 and 13 provide interpretation guidance, but no protein-specific conclusion or comparison with 03. Observations remain to be written.